# 01 - Exploratory Data Analysis: AIS Ship Data

**Goal:** understand the raw AIS data before building the ETA / trajectory model.

Dataset: ~30M AIS position reports (Norwegian west coast, Feb/Jul 2020), split across 40 Parquet files in `data/parquets/`. Each file is roughly one day.

We use the helpers in `src/data_loader.py` so the cleaning logic stays in one place.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# make src/ importable when running from the notebooks/ folder
sys.path.append(str(Path.cwd().parent / "src"))
import data_loader as dl

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 1. Load one day

We explore a single file first (~640k rows). Scale up to `dl.load_sample(n)` or `dl.load_all()` once the analysis is settled.

In [ ]:
print("Available files:", len(dl.list_files()))
df = dl.load_file(0)
print(f"rows: {len(df):,}   vessels: {df['mmsi'].nunique()}")
df.head()

## 2. Schema, dtypes and missing values

Note: the raw columns have **no nulls** — but AIS hides "not available" inside real-looking numbers (see next section).

In [ ]:
df.info()
print("\nRaw nulls per column:")
print(df.isnull().sum())
df.describe()

## 3. AIS sentinel values (the hidden missing data)

AIS transmits special numbers to mean "not available". If we leave them in, the model treats them as real measurements:

| Field | Sentinel | Meaning |
|-------|----------|---------|
| `sog` | `102.3` | speed not available |
| `cog` | `360.0` | course not available |
| `true_heading` | `511` | heading not available |
| `length` | `0` | length unknown |

`dl.clean_ais()` replaces these with `NaN`.

In [ ]:
print("Sentinel counts in raw data:")
print("  sog == 102.3      :", (df['sog'] == dl.SOG_NA).sum())
print("  cog == 360.0      :", (df['cog'] == dl.COG_NA).sum())
print("  true_heading==511 :", (df['true_heading'] == dl.HEADING_NA).sum())
print("  length == 0       :", (df['length'] == dl.LENGTH_NA).sum())

clean = dl.clean_ais(df)
print("\nNulls after clean_ais():")
print(clean[['sog', 'cog', 'true_heading', 'length']].isna().sum())

## 4. Temporal coverage and ping cadence

AIS pings arrive every few seconds. The reference paper resamples to 5-second steps; understanding the native cadence tells us how much to downsample.

In [ ]:
print("time range:", df['date_time_utc'].min(), "->", df['date_time_utc'].max())

# pings-per-vessel distribution
pings = df['mmsi'].value_counts()
print(f"\npings per vessel: median={pings.median():.0f}, max={pings.max():,}")

# inter-ping interval for the busiest vessel
busiest = pings.index[0]
track = dl.get_vessel_track(df, busiest)
dt = track['date_time_utc'].diff().dt.total_seconds().dropna()
print(f"busiest vessel {busiest}: {len(track):,} pings, "
      f"median interval {dt.median():.0f}s")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
pings.plot(kind='hist', bins=40, ax=ax[0], title='Pings per vessel')
dt[dt < 30].plot(kind='hist', bins=30, ax=ax[1], title='Inter-ping interval (s), busiest vessel')
plt.tight_layout()

## 5. Spatial coverage

Plot positions to see the geography and traffic lanes. Colour by speed to reveal where vessels are stationary (ports/anchorages) vs transiting.

In [ ]:
# subsample for a responsive scatter plot
plot_df = clean.sample(min(50_000, len(clean)), random_state=42)

fig, ax = plt.subplots(figsize=(9, 8))
sc = ax.scatter(plot_df['lon'], plot_df['lat'], c=plot_df['sog'],
                s=2, cmap='viridis', vmin=0, vmax=20)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('AIS positions coloured by Speed Over Ground (knots)')
plt.colorbar(sc, ax=ax, label='SOG (kn)')

## 6. Distributions of the key dynamic factors

The paper found **SOG** and **COG** to be the most important predictors after position. Inspect their distributions (sentinels removed).

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
clean['sog'].dropna().plot(kind='hist', bins=60, ax=ax[0], title='SOG (knots)')
clean['cog'].dropna().plot(kind='hist', bins=60, ax=ax[1], title='COG (degrees)')
clean['length'].dropna().plot(kind='hist', bins=40, ax=ax[2], title='Vessel length (m)')
plt.tight_layout()

# a large share of pings are near-stationary (sog ~ 0): vessels in port / anchored
frac_stationary = (clean['sog'].fillna(0) < 0.5).mean()
print(f"Fraction of pings with SOG < 0.5 kn (near-stationary): {frac_stationary:.1%}")

## 7. Navigational status — detecting arrivals

`nav_status == 5` (**moored**) is our signal that a vessel has berthed. Transitions into status 5 mark arrival events, which is how we will build the ETA target label later.

In [ ]:
counts = df['nav_status'].value_counts().sort_index()
counts.index = [f"{c}: {dl.NAV_STATUS.get(c, '?')}" for c in counts.index]
print(counts)

moored = df[df['nav_status'] == 5]
print(f"\nMoored pings: {len(moored):,} across {moored['mmsi'].nunique()} vessels")

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(clean['lon'], clean['lat'], s=1, color='lightgray', alpha=0.3, label='all pings')
ax.scatter(moored['lon'], moored['lat'], s=8, color='red', label='moored (nav_status=5)')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Moored locations = candidate berths / ports')
ax.legend()

## 8. A single vessel trajectory

Sanity-check that one vessel's pings form a coherent track over time — this is the sequence the Bi-GRU model will consume.

In [ ]:
# pick a vessel that actually moves (covers some distance)
movers = clean.groupby('mmsi').agg(
    span_lon=('lon', lambda s: s.max() - s.min()),
    span_lat=('lat', lambda s: s.max() - s.min()),
    n=('lon', 'size'),
)
movers['span'] = movers['span_lon'] + movers['span_lat']
example_mmsi = movers[movers['n'] > 500]['span'].idxmax()

trk = dl.get_vessel_track(clean, example_mmsi)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sc = ax[0].scatter(trk['lon'], trk['lat'], c=trk['sog'], s=6, cmap='plasma')
ax[0].set_title(f'Track of vessel {example_mmsi}')
ax[0].set_xlabel('Longitude'); ax[0].set_ylabel('Latitude')
plt.colorbar(sc, ax=ax[0], label='SOG (kn)')
ax[1].plot(trk['date_time_utc'], trk['sog'])
ax[1].set_title('Speed over time')
ax[1].set_xlabel('time'); ax[1].set_ylabel('SOG (kn)')
plt.tight_layout()

## 9. Takeaways & next steps

- Raw data is clean of nulls but contains AIS sentinels (`sog=102.3`, `cog=360`, `heading=511`, `length=0`) — always run `dl.clean_ais()`.
- Pings are very high frequency (~3 s); we will **resample to a fixed step** (e.g. 5 s, as in the paper) per vessel.
- A large fraction of pings are near-stationary — vessels sitting in port/anchorage.
- `nav_status == 5` (moored) gives us **arrival events** to construct the ETA target.

**Next (`feature_engineer.py`):** group by `mmsi`, resample to fixed steps, build `[lat, lon, sog, cog]` sequences, and derive the time-to-arrival label from moored transitions.